In [ ]:
import pandas as pd
import numpy as np
import ast
from sklearn.cluster import SpectralCoclustering
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 1. Load the dataset
df = pd.read_csv('XWines_Slim_1K_wines_enriched.csv')

# Safe eval for lists stored as strings in the CSV
def parse_list(x):
    try:
        return ast.literal_eval(x)
    except:
        return []

df['Harmonize'] = df['Harmonize'].apply(parse_list)
df['Grapes'] = df['Grapes'].apply(parse_list)

# 2. Bin ABV into categories to make it a categorical feature
df['ABV_Bin'] = pd.cut(df['ABV'], bins=[0, 11, 13.5, 100], labels=['Low_ABV', 'Medium_ABV', 'High_ABV'])

# Explode on Harmonize to make it easier to link each food to the wine features
df_exploded = df.explode('Harmonize').dropna(subset=['Harmonize'])

# Identify top 30 grapes to keep the matrix dense and meaningful
all_grapes = pd.Series([g for sublist in df['Grapes'] for g in sublist])
top_grapes = set(all_grapes.value_counts().head(30).index)

In [ ]:


# 3. Create a mapping of Wine Features vs. Food Pairings
features_list = []
for idx, row in df_exploded.iterrows():
    food = row['Harmonize']
    # Add Type
    features_list.append((food, f"Type_{row['Type']}"))
    # Add Body
    features_list.append((food, f"Body_{row['Body']}"))
    # Add Acidity
    features_list.append((food, f"Acidity_{row['Acidity']}"))
    # Add binned ABV
    features_list.append((food, str(row['ABV_Bin'])))
    # Add Grapes (if in the top 30)
    for g in row['Grapes']:
        if g in top_grapes:
            features_list.append((food, f"Grape_{g}"))

# Create a co-occurrence matrix (cross-tabulation)
df_features = pd.DataFrame(features_list, columns=['Food', 'Feature'])
cooc = pd.crosstab(df_features['Food'], df_features['Feature'])

# 4. Run Spectral Coclustering
n_clusters = 5
model = SpectralCoclustering(n_clusters=n_clusters, random_state=42)
model.fit(cooc)

# Rearrange matrix according to the biclusters for visualization
fit_data = cooc.values[np.argsort(model.row_labels_)]
fit_data = fit_data[:, np.argsort(model.column_labels_)]

row_labels = cooc.index[np.argsort(model